# WSF Tracker — Year-Code Sensitivity Analysis

> **STATUS: TODO — this notebook has not been run. Complete Step 1 first to confirm the year encoding, then update `YEAR_CODE_MAP` in Step 2 before running the sensitivity loop.**

**Purpose:** Re-run the raster built-up validation for WSF Tracker across a range of `as_of_code` cutoffs (covering 2019–2026 in whatever encoding the raster uses) to assess how much accuracy metrics depend on the chosen year cutoff.

**Why this matters:** The WSF Tracker encodes settlement detection year as an integer. The `as_of_code` value in `configs/validation_configs.yaml` determines which pixels are counted as built-up. Two encodings are possible:
- *Sequential (1-indexed from 2015):* 1=2015, 2=2016, …, 9=2023, 10=2024
- *Two-digit year:* 15=2015, 16=2016, …, 23=2023, 24=2024

Under the sequential encoding, `as_of_code: 19` exceeds the max code and captures all years — the cutoff is irrelevant. Under the two-digit encoding, `as_of_code: 19` means 2019 and misses 4–5 years of growth. **Step 1 resolves this ambiguity.**

**Outputs:**
- `outputs/wsf_year_sensitivity.csv` — per-city metrics for each year cutoff
- Summary table printed inline: mean F1 and mean signed area bias per cutoff year
- Flag if any cutoff deviates >0.05 F1 from the baseline (`as_of_code: 19`)

Created by: Caroline Gevaert — The World Bank  
Financed by: The Gates Foundation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.windows import from_bounds
from rasterio.transform import Affine
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling
import yaml
import warnings

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation"
)
print("PROJECT_ROOT:", PROJECT_ROOT)

## Step 1 — Confirm the WSF year encoding

> **TODO: Run this cell, read the printed unique values, and update `YEAR_CODE_MAP` in the next cell.**

This reads the WSF raster for one city (ssd-juba) and prints every non-zero unique pixel value present. Use the output to determine whether the encoding is sequential (max ≈ 9–10) or two-digit year (max ≈ 23–24).

In [ ]:
# TODO: update this path to the actual WSF raster for ssd-juba
WSF_SAMPLE_PATH = PROJECT_ROOT / "data/01_raw/ssd-juba/predicted/raster/wsf-tracker/wsf_tracker_ssd_juba.tif"

if not WSF_SAMPLE_PATH.exists():
    raise FileNotFoundError(
        f"WSF raster not found at:\n  {WSF_SAMPLE_PATH}\n"
        "Update WSF_SAMPLE_PATH above to the correct location."
    )

with rasterio.open(WSF_SAMPLE_PATH) as src:
    # Read a manageable sample if the raster is large (first overview or full read)
    arr = src.read(1)
    nodata = src.nodata
    print(f"Raster shape : {arr.shape}")
    print(f"Raster dtype : {arr.dtype}")
    print(f"Nodata value : {nodata}")

# Unique non-zero, non-nodata values
mask = arr != 0
if nodata is not None:
    mask &= (arr != nodata)
unique_vals = np.unique(arr[mask])

print(f"\nUnique non-zero pixel values ({len(unique_vals)} total):")
print(unique_vals)
print(f"\nMin non-zero: {unique_vals.min()}  |  Max non-zero: {unique_vals.max()}")

print("\n--- Interpretation guide ---")
print("If max ≈ 9–10  → sequential encoding (1=2015, 2=2016, …)")
print("If max ≈ 23–24 → two-digit year encoding (15=2015, 16=2016, …)")

In [ ]:
# TODO: fill in YEAR_CODE_MAP based on Step 1 output above.
#
# Example A — sequential encoding (1-indexed from 2015):
#   YEAR_CODE_MAP = {1: 2015, 2: 2016, 3: 2017, 4: 2018, 5: 2019,
#                    6: 2020, 7: 2021, 8: 2022, 9: 2023, 10: 2024}
#
# Example B — two-digit year encoding:
#   YEAR_CODE_MAP = {15: 2015, 16: 2016, 17: 2017, 18: 2018, 19: 2019,
#                    20: 2020, 21: 2021, 22: 2022, 23: 2023, 24: 2024}
#
YEAR_CODE_MAP = {}  # <-- FILL IN after running Step 1

if not YEAR_CODE_MAP:
    raise ValueError(
        "YEAR_CODE_MAP is empty. Run Step 1 first and fill in the mapping above."
    )

# Derive the sensitivity sweep: one as_of_code per year from 2019 to
# the last year present in the data (inclusive)
code_to_year = YEAR_CODE_MAP                           # {code: year}
year_to_code = {v: k for k, v in code_to_year.items()}  # {year: code}

START_YEAR = 2019
END_YEAR   = max(code_to_year.values())

SWEEP_YEARS = [y for y in range(START_YEAR, END_YEAR + 1) if y in year_to_code]
SWEEP_CODES = [year_to_code[y] for y in SWEEP_YEARS]

print("Year sweep:")
for year, code in zip(SWEEP_YEARS, SWEEP_CODES):
    print(f"  as_of_code={code:3d}  →  {year}")

BASELINE_CODE = year_to_code.get(2019)   # current config value's year
print(f"\nBaseline (as_of_code: 19 in config) maps to: "
      f"{code_to_year.get(19, 'NOT IN MAP — value 19 exceeds data range')}")

## Step 2 — Sensitivity loop

> **TODO: Run after completing Step 1 and setting `YEAR_CODE_MAP`.**

For each `as_of_code` in the sweep, re-runs the WSF validation across all cities in the AOI tracker and records per-city metrics.

In [ ]:
# ---- Shared helpers (subset of 02_builtup_accuracy_rasters.ipynb) ----

def _pixel_area_from_transform(transform) -> float:
    return float(abs(transform.a * transform.e))

def open_in_target_crs(src, target_crs: str):
    if src.crs is None:
        raise ValueError("Raster has no CRS.")
    if str(src.crs) == str(target_crs):
        return src
    return WarpedVRT(src, crs=target_crs, resampling=Resampling.nearest)

def rasterize_ref_fraction(ref_geoms, out_shape, transform, oversample=4) -> np.ndarray:
    if len(ref_geoms) == 0:
        return np.zeros(out_shape, dtype="float32")
    if oversample <= 1:
        mask = features.rasterize(
            [(g, 1) for g in ref_geoms], out_shape=out_shape,
            transform=transform, fill=0, dtype="uint8"
        )
        return mask.astype("float32")
    h, w = out_shape
    oh, ow = h * oversample, w * oversample
    hi_transform = transform * Affine.scale(1.0 / oversample, 1.0 / oversample)
    hi = features.rasterize(
        [(g, 1) for g in ref_geoms], out_shape=(oh, ow),
        transform=hi_transform, fill=0, dtype="uint8"
    ).astype("float32")
    return hi.reshape(h, oversample, w, oversample).mean(axis=(1, 3)).astype("float32")

def aoi_mask_for_window(aoi_geom, out_shape, transform) -> np.ndarray:
    mask = features.rasterize(
        [(aoi_geom, 1)], out_shape=out_shape,
        transform=transform, fill=0, dtype="uint8"
    )
    return mask.astype(bool)

def wsf_built_pixels(arr: np.ndarray, as_of_code: int,
                     built_value_min: int = 1, nonbuilt_value: int = 0) -> np.ndarray:
    """Binary mask: pixels first detected at or before as_of_code."""
    return (arr != nonbuilt_value) & (arr >= built_value_min) & (arr <= as_of_code)

In [ ]:
# ---- Load AOI tracker to enumerate cities ----
# TODO: update paths if your layout differs from the standard project structure

CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"
TRACKER_PATH = PROJECT_ROOT / "data/02_interim/aoi_tracker.csv"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

ROOT      = PROJECT_ROOT / cfg["root_dir"]
DATA_DIR  = ROOT / cfg.get("data_dir", "data/01_raw")
CRS       = cfg["crs"]
TAU_FRAC  = float(cfg.get("raster", {}).get("preprocessing", {}).get("tau_frac", 0.2))
OVERSAMPLE = int(cfg.get("raster", {}).get("preprocessing", {}).get("oversample_factor", 4))

# WSF-specific config from validation_configs.yaml
wsf_cfg = next(
    (d for d in cfg["raster"]["datasets"] if d["name"] == "wsf-tracker"),
    None
)
if wsf_cfg is None:
    raise ValueError("No 'wsf-tracker' entry found under raster.datasets in validation_configs.yaml")

WSF_BUILT_MIN  = int(wsf_cfg["binarize"].get("built_value_min", 1))
WSF_NONBUILT   = int(wsf_cfg["binarize"].get("nonbuilt_value", 0))

tracker = pd.read_csv(TRACKER_PATH, dtype=str)
tracker.columns = tracker.columns.str.strip()
tracker = tracker.apply(lambda c: c.str.strip() if c.dtype == object else c)
tracker = tracker[tracker["Suitable (yes/N)"].str.lower() == "yes"]

print(f"Cities in tracker (suitable): {tracker['Dataset code'].nunique()}")
print(f"CRS: {CRS} | TAU_FRAC: {TAU_FRAC} | OVERSAMPLE: {OVERSAMPLE}")
print(f"WSF built_value_min: {WSF_BUILT_MIN} | nonbuilt_value: {WSF_NONBUILT}")

In [ ]:
# ---- Per-city, per-cutoff evaluation ----

def eval_wsf_for_city(city: str, row: pd.Series, as_of_code: int) -> dict | None:
    """
    Run WSF validation for one city at one as_of_code cutoff.
    Returns a dict of city-level metrics, or None if data is missing.
    """
    folder   = row["dataset_folder_name"]
    aoi_file = row.get("aoi_file_name", "").strip()
    ref_file = row.get("reference_file_name", "").strip()

    aoi_path = DATA_DIR / folder / "aoi"    / aoi_file
    ref_path = DATA_DIR / folder / "vector" / ref_file

    # TODO: update this pattern to match the actual WSF raster filename convention
    wsf_dir  = DATA_DIR / folder / "raster" / "wsf-tracker"
    wsf_candidates = list(wsf_dir.glob("*.tif")) if wsf_dir.exists() else []

    if not aoi_path.exists():
        warnings.warn(f"{city}: AOI not found — {aoi_path}")
        return None
    if not ref_path.exists():
        warnings.warn(f"{city}: reference not found — {ref_path}")
        return None
    if not wsf_candidates:
        warnings.warn(f"{city}: no WSF raster found in {wsf_dir}")
        return None

    wsf_path = wsf_candidates[0]

    # Load AOI + reference
    aoi = gpd.read_file(aoi_path).to_crs(CRS)
    aoi_union = aoi.geometry.union_all()

    ref = gpd.read_file(ref_path).to_crs(CRS)
    ref_sindex = ref.sindex

    # Tiles (1 km × 1 km)
    from shapely.geometry import box
    minx, miny, maxx, maxy = aoi_union.bounds
    tile_size = float(cfg.get("vector", {}).get("preprocessing", {}).get("tile_size_m", 1000))
    xs = np.arange(minx, maxx, tile_size)
    ys = np.arange(miny, maxy, tile_size)
    tiles = [
        box(x, y, x + tile_size, y + tile_size)
        for x in xs for y in ys
        if box(x, y, x + tile_size, y + tile_size).intersects(aoi_union)
    ]

    tp_m2 = fp_m2 = fn_m2 = valid_area = 0.0

    with rasterio.open(wsf_path) as src:
        ds = open_in_target_crs(src, CRS)
        nodata = ds.nodata

        for tile_geom in tiles:
            win = from_bounds(*tile_geom.bounds, transform=ds.transform)
            if win.width <= 0 or win.height <= 0:
                continue

            out_shape = (int(round(win.height)), int(round(win.width)))
            arr = ds.read(1, window=win, boundless=True,
                          fill_value=nodata if nodata is not None else 0)
            transform = rasterio.windows.transform(win, ds.transform)
            pixel_area = _pixel_area_from_transform(transform)

            aoi_mask = aoi_mask_for_window(aoi_union, arr.shape, transform)
            valid = aoi_mask.copy()
            if nodata is not None:
                valid &= (arr != nodata)

            n_valid = int(valid.sum())
            if n_valid == 0:
                continue

            pred_bin = wsf_built_pixels(arr, as_of_code, WSF_BUILT_MIN, WSF_NONBUILT)

            possible = list(ref_sindex.intersection(tile_geom.bounds))
            ref_tile = ref.iloc[possible]
            ref_tile = ref_tile[ref_tile.intersects(tile_geom)]
            f_ref = rasterize_ref_fraction(
                list(ref_tile.geometry), arr.shape, transform, oversample=OVERSAMPLE
            )
            ref_bin = (f_ref >= TAU_FRAC)

            p = pred_bin[valid]
            r = ref_bin[valid]
            tp_m2    += float(np.logical_and(p, r).sum()) * pixel_area
            fp_m2    += float(np.logical_and(p, ~r).sum()) * pixel_area
            fn_m2    += float(np.logical_and(~p, r).sum()) * pixel_area
            valid_area += n_valid * pixel_area

    if valid_area == 0:
        return None

    precision = tp_m2 / (tp_m2 + fp_m2) if (tp_m2 + fp_m2) > 0 else 0.0
    recall    = tp_m2 / (tp_m2 + fn_m2) if (tp_m2 + fn_m2) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "city":                city,
        "as_of_code":          as_of_code,
        "as_of_year":          code_to_year.get(as_of_code, None),
        "precision_area":      round(precision, 4),
        "recall_area":         round(recall, 4),
        "f1_area":             round(f1, 4),
        "tp_m2":               round(tp_m2, 1),
        "fp_m2":               round(fp_m2, 1),
        "fn_m2":               round(fn_m2, 1),
        "signed_area_bias_m2": round(fp_m2 - fn_m2, 1),
        "valid_area_m2":       round(valid_area, 1),
    }

In [ ]:
# ---- Main sweep ----
# TODO: Run after Step 1 is complete and YEAR_CODE_MAP is filled in.

results = []
cities_grouped = list(tracker.groupby("Dataset code"))

for year, code in zip(SWEEP_YEARS, SWEEP_CODES):
    print(f"\n=== as_of_code={code} ({year}) ===")
    for city, group in cities_grouped:
        # Use first row per city (one AOI/reference per city for raster validation)
        row = group.iloc[0]
        try:
            res = eval_wsf_for_city(city, row, as_of_code=code)
            if res:
                results.append(res)
                print(f"  {city}: F1={res['f1_area']:.3f}  bias={res['signed_area_bias_m2']/1e6:.1f} km²")
            else:
                print(f"  {city}: SKIPPED (missing data)")
        except Exception as e:
            print(f"  {city}: ERROR — {e}")

sensitivity_df = pd.DataFrame(results)

out_path = PROJECT_ROOT / "outputs/wsf_year_sensitivity.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
sensitivity_df.to_csv(out_path, index=False)
print(f"\nResults saved → {out_path}")
print(f"Rows: {len(sensitivity_df)}  |  Cities: {sensitivity_df['city'].nunique()}  |  Cutoffs: {sensitivity_df['as_of_year'].nunique()}")

## Step 3 — Summary table and flagging

> **TODO: Run after the sensitivity loop completes.**

In [ ]:
# ---- Summary: mean F1 and mean bias per cutoff year ----

summary = (
    sensitivity_df
    .groupby(["as_of_year", "as_of_code"])
    .agg(
        n_cities             =("city",                "nunique"),
        mean_f1              =("f1_area",             "mean"),
        median_f1            =("f1_area",             "median"),
        mean_precision       =("precision_area",      "mean"),
        mean_recall          =("recall_area",         "mean"),
        mean_bias_km2        =("signed_area_bias_m2", lambda x: x.mean() / 1e6),
    )
    .reset_index()
    .sort_values("as_of_year")
)

# Flag relative to baseline (as_of_code: 19)
baseline_row = summary[summary["as_of_code"] == 19]
if len(baseline_row) == 1:
    baseline_f1 = float(baseline_row["mean_f1"].iloc[0])
    summary["f1_delta_vs_baseline"] = (summary["mean_f1"] - baseline_f1).round(4)
    summary["flagged"] = summary["f1_delta_vs_baseline"].abs() > 0.05
    print(f"Baseline (as_of_code=19) mean F1: {baseline_f1:.4f}")
else:
    summary["f1_delta_vs_baseline"] = np.nan
    summary["flagged"] = False
    print("Note: as_of_code=19 not in sweep (year 2019 may not be in data range).")

print("\n=== WSF Year-Code Sensitivity Summary ===")
display(
    summary[[
        "as_of_year", "as_of_code", "n_cities",
        "mean_f1", "median_f1", "mean_precision", "mean_recall",
        "mean_bias_km2", "f1_delta_vs_baseline", "flagged"
    ]].round(4)
)

flagged = summary[summary["flagged"]]
if len(flagged):
    print(f"\n⚠  {len(flagged)} cutoff(s) differ from baseline by >0.05 F1:")
    for _, r in flagged.iterrows():
        print(f"   {int(r.as_of_year)} (code={int(r.as_of_code)}): "
              f"mean F1={r.mean_f1:.4f}  delta={r.f1_delta_vs_baseline:+.4f}")
else:
    print("\n✓  No cutoff differs from baseline by more than 0.05 F1.")

In [ ]:
# ---- Interpretation: is as_of_code=19 capturing all years or cutting off early? ----

max_code_in_data = max(code_to_year.keys())
max_year_in_data = code_to_year[max_code_in_data]

print("=" * 55)
print("as_of_code=19 interpretation")
print("=" * 55)

if 19 > max_code_in_data:
    print(f"as_of_code=19 EXCEEDS the maximum code in the data ({max_code_in_data} = {max_year_in_data}).")
    print("→ The current config captures ALL years — the cutoff is effectively the final year of data.")
    print("→ No temporal bias is introduced by the current as_of_code value.")
elif 19 in code_to_year:
    mapped_year = code_to_year[19]
    years_missing = [y for y in range(mapped_year + 1, max_year_in_data + 1)]
    print(f"as_of_code=19 maps to {mapped_year} in the confirmed encoding.")
    print(f"→ Years {years_missing} are EXCLUDED from the built-up mask.")
    print(f"→ Buildings added {mapped_year+1}–{max_year_in_data} appear as False Negatives.")
    print("→ Update as_of_code in configs/validation_configs.yaml to match the reference year.")
else:
    print(f"as_of_code=19 is not a valid code in the confirmed mapping.")
    print(f"Valid codes: {sorted(code_to_year.keys())}")
    print("→ Check the encoding and update as_of_code accordingly.")